## 1. Import Libraries

In [1]:
import os
import numpy as np
import pandas as pd
from functools import reduce

# Reproducibility
RANDOM_SEED = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Pipeline & Metrics
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer

# Model
from sklearn.ensemble import RandomForestRegressor

## 2. Load and Merge Data

In [2]:
companyInfo   = pd.read_csv("company_info.csv")
monashIndex   = pd.read_csv("monashIndex.csv")
stockData     = pd.read_csv("stock_data.csv")
training      = pd.read_csv("training_targets.csv")
funds         = pd.read_csv("fed_funds_rate.csv")
inflation     = pd.read_csv("fed_inflation_rate.csv")
unemployment  = pd.read_csv("fed_unemployment_rate.csv")
fiveyearT     = pd.read_csv("us_5_year_treasury.csv")
tenyearT      = pd.read_csv("us_10_year_treasury.csv")
vixIndex      = pd.read_csv("vix_index.csv")
test_ids      = pd.read_csv("testing_targets.csv")  

In [3]:
# From the EDA, we identified two redundant attributes ('revenue_tier' and 'business_model') 
# due to their high correlation with other variables, so they are removed.
# For stock data, all missing values occur only in the first records of stocks listed after 2020_01,
# making them non-informative. These rows are therefore dropped

companyInfo = companyInfo.drop(columns=[c for c in ['revenue_tier', 'business_model'] if c in companyInfo.columns])
stockData = stockData.dropna(axis=0, how='any')

# Merge dataset
df1_parts = [monashIndex, funds, inflation, unemployment, fiveyearT, tenyearT, vixIndex, training]
df1 = reduce(lambda left, right: pd.merge(left, right, on="month_id", how="left"), df1_parts)

df2 = pd.merge(companyInfo, stockData, on=["stock_id"], how="left")
df_final = pd.merge(df2, df1, on=["month_id","stock_id"], how="left")

# Reorder columns (keep month_id first)
df_final = df_final[['month_id'] + [c for c in df_final.columns if c != 'month_id']]

## 3. Time Handling and Leakage-Safe Target Setup

In [4]:
def to_period(mstr: str) -> pd.Period:
    """Convert 'YYYY_MM' or 'YYYY-MM' 
    to a pandas monthly Period (e.g., '2020-01')."""
    
    y, m = str(mstr).replace('-', '_').split('_')
    return pd.Period(f"{int(y):04d}-{int(m):02d}", freq='M')

# Make a working copy and ensure consistent ID formats
df = df_final.copy()
for d in (df, test_ids):
    d['stock_id'] = d['stock_id'].astype(str)
    d['month_id'] = d['month_id'].astype(str)

# Attach a proper monthly Period index for time-series alignment
df['__period'] = df['month_id'].apply(to_period)
df.sort_values(['stock_id', '__period'], inplace=True, ignore_index=True)

# Regression target: next-month excess return
if 'excess_return' not in df.columns:
    raise ValueError("Column 'excess_return' missing from data.")
df['y_next_reg'] = df.groupby('stock_id')['excess_return'].shift(-1)

# Simple safe lags from return-related contemporaneous columns
for col in ['index_return', 'excess_return']:
    if col in df.columns:
        df[f'lag1_{col}'] = df.groupby('stock_id')[col].shift(1)

## 4. Build Leakage-Safe Train Frame & Feature Set

In [5]:
# Keep only rows that have NEXT-month targets available
train_reg = df[~df['y_next_reg'].isna()].copy()

# Columns we must NOT feed to the model (answers/meta/contemporaneous returns)
block_cols = {
    # same-month “answer-like” info (would leak the truth)
    'intramonth_return', 'index_return', 'excess_return',
    # engineered targets and identifiers / time index
    'y_next_reg', 'stock_id', 'month_id', '__period'
}

In [6]:
# Feature matrix 
# Define the feature schema from train_cls, excluding blocked columns
X_reg = train_reg.drop(columns=[c for c in train_reg.columns if c in block_cols])
y_reg = train_reg['y_next_reg'].astype(float)

In [7]:
# Data types for preprocessing 
# Numeric features (everything non-object); categoricals will be handled separately
numeric_cols = X_reg.select_dtypes(exclude='object').columns.tolist()

# Ordinal categories (from business logic). We only keep those present in X_full.
ORDINAL_MAPPING = [
    ['Mature', 'Growth', 'Cyclical'],                        # business_maturity
    ['Market_Leader', 'Strong_Player', 'Niche_Specialist'],  # competitive_position
    ['Large', 'Mid', 'Small'],                               # market_cap_category
    ['High_Margin', 'Standard', 'Low_Margin'],               # profitability_profile
    ['Capital_Intensive', 'Moderate', 'Asset_light'],        # asset_intensity
    ['Strong', 'Stable', 'Developing']                       # financial_strength
]
ORDINAL_COLS = [
    'business_maturity', 'competitive_position', 'market_cap_category',
    'profitability_profile', 'asset_intensity', 'financial_strength'
]
NOMINAL_COLS = ['sector', 'geographic_focus']

# only keep columns that actually exist in the current feature set
ORDINAL_COLS = [c for c in ORDINAL_COLS if c in X_reg.columns]
NOMINAL_COLS = [c for c in NOMINAL_COLS if c in X_reg.columns]


## 5. Pipeline Preprocessing

In [8]:
# Numeric pipeline
# Median imputation (robust to skew/outliers) + Standard scaling (mean=0, std=1)
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Ordinal pipeline 
# Impute missing categories with most frequent, then apply OrdinalEncoder.
# Category order comes from domain knowledge (ORDINAL_MAPPING).
ordinal_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=ORDINAL_MAPPING[:len(ORDINAL_COLS)],  
        handle_unknown='use_encoded_value',              
        unknown_value=-1                                 
    ))
])

# Nominal pipeline 
# Impute with most frequent, then expand categories into one-hot vectors.
# Unseen categories are ignored (keeps schema stable across train/test).
nominal_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


In [9]:
# Combine everything
# ColumnTransformer applies the right pipeline to each feature subset.
preprocess = ColumnTransformer(
    transformers=[
        ('num',  num_pipe,     numeric_cols),  # continuous features
        ('ord',  ordinal_pipe, ORDINAL_COLS),  # business-logic order
        ('nom',  nominal_pipe, NOMINAL_COLS)   # categorical (no inherent order)
    ],
    remainder='drop'  
)

## 6. Final Model Configuration

In [10]:
# Regression Model
# Ridge regression is used to predict continuous excess returns.
# It applies L2 regularization, which stabilizes coefficients
# when predictors are correlated (common in financial data).
# RidgeCV automatically selects the best alpha via cross-validation.
# - Search space: log-spaced alphas from 1e-3 to 1e+3
# - Cross-validation: 5-fold TimeSeriesSplit (no shuffling, leakage-safe)
tscv = TimeSeriesSplit(n_splits=5)
reg = Pipeline([
    ('preprocess', preprocess),
    ('model', RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=RANDOM_SEED,
        n_jobs=-1
    ))
])


## 7. Cross-Validated Evaluation

In [11]:
# At this stage we are NOT tuning hyperparameters anymore.
# GridSearchCV (with TSCV) was used earlier during model selection.
# Here, we simply evaluate the final models to report
# their reproducible performance using TimeSeriesSplit.
# Regression: RMSE
cv_reg = cross_validate(
    reg, X_reg, y_reg,
    cv=tscv,
    scoring={'rmse': 'neg_root_mean_squared_error'},  # sklearn outputs neg RMSE
    n_jobs=-1,
    return_train_score=False
)
mean_rmse = -cv_reg['test_rmse'].mean()  
print(f"RMSE (Regression Task):         {mean_rmse:.6f}")


RMSE (Regression Task):         0.097396


## 8. Fit Final Model

In [12]:
# After cross-validated evaluation, we now fit each model
# on the entire training dataset. This ensures the models
# learn from all available information before generating
# predictions for the test set (testing_targets.csv).

# Regression
reg.fit(X_reg, y_reg)

,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('ord', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## 9. Prediction on _Testing_target.csv_

In [13]:
def build_snapshot(df_all: pd.DataFrame, feature_cols, cutoff_month: str) -> pd.DataFrame:
    """
    For each stock_id, select the last available row strictly BEFORE `cutoff_month`,
    and return only ['stock_id'] + feature_cols. This guarantees we never peek
    into the future when making July-2023 predictions.
    """
    tmp = df_all.copy()
    tmp['__period'] = tmp['month_id'].apply(to_period)
    cutoff = to_period(cutoff_month)
    hist = tmp[tmp['__period'] < cutoff].sort_values(['stock_id', '__period'])
    last_rows = hist.groupby('stock_id', as_index=False).tail(1)
    return last_rows[['stock_id'] + feature_cols].copy()

In [14]:
# Use the same feature columns as training
feature_cols = X_reg.columns.tolist()

# Build snapshots for the cutoff '2023_07' (as per your earlier script)
snapshot = build_snapshot(df, feature_cols, '2023_07')

# Align features for prediction (add missing columns as needed)
to_pred = test_ids[['stock_id', 'month_id']].merge(snapshot, on='stock_id', how='left')

# If any feature columns are missing after the merge, create placeholders.
# - numeric being NaN (will be imputed by numeric/ordinal pipelines)
# - categorical being 'unknown' (OneHot ignores unseen; Ordinal maps unknown_value=-1)
for col in feature_cols:
    if col not in to_pred.columns:
        to_pred[col] = np.nan if col in numeric_cols else 'unknown'
        
# Keep only the model feature columns in the right order        
X_input = to_pred[feature_cols]

In [15]:
# Predict
pred_excess  = reg.predict(X_input).astype(float)

## 10. Overwrite & Save Prediction

In [16]:
result_pred = test_ids.copy()
result_pred['excess_return']     = pred_excess
result_pred.to_csv('testing_targets.csv', index=False)